In [54]:
import kagglehub

path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
print(path)

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
/kaggle/input/fashion-product-images-small


In [34]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [35]:
import os

print(os.listdir(path))

['myntradataset', 'images', 'styles.csv']


In [36]:
import pandas as pd
import os

csv_path = os.path.join(path, "styles.csv")

df = pd.read_csv(csv_path, on_bad_lines='skip')
df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [37]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

(44424, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  object 
 2   masterCategory      44424 non-null  object 
 3   subCategory         44424 non-null  object 
 4   articleType         44424 non-null  object 
 5   baseColour          44409 non-null  object 
 6   season              44403 non-null  object 
 7   year                44423 non-null  float64
 8   usage               44107 non-null  object 
 9   productDisplayName  44417 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 3.4+ MB
None
id                      0
gender                  0
masterCategory          0
subCategory             0
articleType             0
baseColour             15
season                 21
year                    1
usage           

In [38]:
df = df.dropna()

In [39]:
df = df.drop_duplicates()

In [40]:
df = df[['id', 'masterCategory']]

In [41]:
IMAGE_DIR = "images/"

df['image_path'] = df['id'].astype(str) + ".jpg"
df['image_path'] = df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))

In [42]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['masterCategory'])

num_classes = len(le.classes_)
print(num_classes)

7


In [43]:
# Step 1: Remove rare classes
class_counts = df['label'].value_counts()
df = df[df['label'].isin(class_counts[class_counts >= 5].index)]

# Step 2: Split safely
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [44]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [45]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="masterCategory",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="masterCategory",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

Found 0 validated image filenames belonging to 0 classes.
Found 0 validated image filenames belonging to 0 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:918: UserWarning: Found 35260 invalid image filename(s) in x_col="image_path". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:918: UserWarning: Found 8816 invalid image filename(s) in x_col="image_path". These filename(s) will be ignored.
  warnings.warn(


In [46]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # Freeze base model

In [47]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [48]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [49]:
import pandas as pd
import tensorflow as tf
import os
from PIL import Image
import numpy as np

In [50]:
df = pd.read_csv(os.path.join(path, "styles.csv"), on_bad_lines='skip')

In [51]:
def load_image(img_id):
    img_path = os.path.join(path, "images", str(img_id) + ".jpg")
    img = Image.open(img_path).resize((224, 224))
    img = np.array(img) / 255.0
    return img

In [52]:
print("Train samples:", train_generator.samples)
print("Test samples:", test_generator.samples)

Train samples: 0
Test samples: 0
